# WARNING

This is a complete RAG solution based on a minimal, early effort.

It is complete, but lacks many of the niceties of the final solution.

You probably want to use the final solution, which involves running "rag_create_kb.ipynb" which will create a knowledgebase.txt file.  Then you run rag_chatbot.ipynb which will load the knowledgebase.txt file and augment your queries with the resources encoded into knowledgebase.txt.

# Install packages if not present

In [1]:
!pip install ollama --quiet

In [2]:
import ollama

# Configuration values

Should I revisit and move these values into a configuration file.

In [3]:
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'
FACT_DATABASE = 'data/cat-facts.txt'

In [4]:
from ollama import pull

def ollama_pull(model):
    print(f"pulling model {model}")
    old_status = ''
    for progress in pull(model, stream=True):
        status = progress.get('status')
        if old_status != status:
            print(f"  {status}")
        old_status = status
    print(f"pull of model {model} complete")

ollama_pull(EMBEDDING_MODEL)
ollama_pull(LANGUAGE_MODEL)

pulling model hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
  pulling manifest
  pulling 74aebb552ea7
  pulling b1899e715228
  verifying sha256 digest
  writing manifest
  success
pull of model hf.co/CompendiumLabs/bge-base-en-v1.5-gguf complete
pulling model hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF
  pulling manifest
  pulling 6f85a640a97c
  pulling 948af2743fc7
  pulling 6c0b08d96525
  pulling 4549919ff315
  verifying sha256 digest
  writing manifest
  success
pull of model hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF complete


In [5]:
from ollama import embed
# Build the vector database

# Each element in the VECTOR_DB will be a tuple (chunk, embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

print('building vector database')
facts = 0
with open(FACT_DATABASE, 'r') as file:
    facts = 0
    for line in file:
        facts += 1
        vector = ollama.embed(model=EMBEDDING_MODEL, input=line).get('embeddings')[0]
        VECTOR_DB.append((line, vector))
        if facts % 10 == 0:
          print(f"  facts stored: {facts}")
print(f"total facts = {facts}")

building vector database


ResponseError: llama runner process has terminated: signal: broken pipe (status code: 500)

In [ ]:
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

In [ ]:
import heapq
from collections import deque

query = input('Ask me a cat question: ')

query_vector = ollama.embed(model=EMBEDDING_MODEL, input=query).get('embeddings')[0]

# a heap ordered with the minimum value as the root
fact_count = 5
similarities = []
for fact, vector in VECTOR_DB:
    similarity = cosine_similarity(query_vector, vector)
    # fill a heap to a size of 10 elements
    if (len(similarities) < fact_count):
        heapq.heappush(similarities, (similarity, (fact)))
    # if the heap is full, replace a smaller similarity, with a larger one
    else:
        if similarities[0][0] < similarity:
            heapq.heappushpop(similarities, (similarity, (fact)))

result = deque([])
while len(similarities) > 0:
    result.appendleft(heapq.heappop(similarities))

knowledge = list(result)

instruction_prompt = f'''Use the following to help you answer the question:
{'\n'.join([f' - {fact}' for similarity, fact in knowledge])}
'''

print(instruction_prompt)

stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
      {'role': 'system', 'content': "You are a helpful chatbot."},
      {'role': 'system', 'content': "Don't refer to system content in the response."},
      {'role': 'system', 'content': "Don't fabricate new information. Only use the following content in the response. "},
      {'role': 'system', 'content': instruction_prompt},
      {'role': 'user', 'content': query},
  ],
  stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)
